In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
base_dir = os.path.expanduser('~/Projects/groundwater/data/henry_data/grid_scenarios_20x40')
# coupling_scenarios_dir = os.path.join(base_dir, 'coupling_scenarios')
# diffusion_scenarios_dir = os.path.join(base_dir, 'diffusion_scenarios')
# os.path.exists(coupling_scenarios_dir), os.path.exists(diffusion_scenarios_dir) 

scenarios_dir = os.path.join(base_dir)
os.path.exists(scenarios_dir)

True

In [9]:
# Build scenario map from the unified scenarios directory layout.
scenarios_root = os.path.join(base_dir, 'scenarios')

def build_scenario_map(root_dir):
    scenario_map = {}
    if not os.path.exists(root_dir):
        return scenario_map

    for scenario in sorted(os.listdir(root_dir)):
        if scenario.startswith('.'):
            continue

        scenario_dir = os.path.join(root_dir, scenario)
        if not os.path.isdir(scenario_dir):
            continue

        scenario_cfg_path = os.path.join(scenario_dir, 'scenario_config.json')
        runs_cfg_path = os.path.join(scenario_dir, 'runs_config.json')

        if not os.path.exists(scenario_cfg_path):
            continue

        with open(scenario_cfg_path, 'r') as f:
            scenario_config = json.load(f)

        runs_config = {'runs': []}
        if os.path.exists(runs_cfg_path):
            with open(runs_cfg_path, 'r') as f:
                runs_config = json.load(f)

        run_records = runs_config.get('runs', [])
        run_meta_by_name = {rec.get('run'): rec for rec in run_records if rec.get('run')}

        runs = []
        for run in sorted(os.listdir(scenario_dir)):
            if run.startswith('.') or not run.startswith('run_'):
                continue

            windows_path = os.path.join(scenario_dir, run, 'windows.npz')
            if os.path.exists(windows_path):
                runs.append(run)

        if runs:
            scenario_map[scenario] = {
                'config': scenario_config,
                'runs': runs,
                'run_meta_by_name': run_meta_by_name,
            }

    return scenario_map

scenario_map = build_scenario_map(scenarios_root)
if not scenario_map:
    raise ValueError(f'No scenarios with windows.npz were found in: {scenarios_root}')

def _sort_key(x):
    try:
        return float(x)
    except Exception:
        return str(x)

# Use a single dropdown to select the run name across all scenarios.
run_sets = [set(v['runs']) for v in scenario_map.values() if v.get('runs')]
common_runs = sorted(set.intersection(*run_sets), key=_sort_key) if run_sets else []
if not common_runs:
    common_runs = sorted(next(iter(scenario_map.values()))['runs'], key=_sort_key)

run_dropdown = widgets.Dropdown(
    options=common_runs,
    value=common_runs[0],
    description='Run:',
    layout=widgets.Layout(width='55%')
)

run_label = widgets.HTML()
plot_output = widgets.Output()

def render_plot(*_):
    selected_run = run_dropdown.value

    records = []
    beta_vals = set()
    diff_vals = set()

    for scenario, info in scenario_map.items():
        cfg = info['config']
        try:
            beta_c = float(cfg.get('beta_c'))
            diffc = float(cfg.get('diffc'))
        except Exception:
            continue

        records.append({
            'scenario': scenario,
            'beta_c': beta_c,
            'diffc': diffc,
            'scenario_index': cfg.get('scenario_index', 'N/A'),
            'info': info,
        })
        beta_vals.add(beta_c)
        diff_vals.add(diffc)

    beta_vals = sorted(beta_vals)
    diff_vals = sorted(diff_vals)

    if not records:
        with plot_output:
            clear_output(wait=True)
            print('No plottable scenarios found.')
        return

    plot_data = {}
    vmax = 0.0

    for rec in records:
        windows_path = os.path.join(scenarios_root, rec['scenario'], selected_run, 'windows.npz')
        if not os.path.exists(windows_path):
            continue

        with np.load(windows_path) as windows_data:
            final_window = windows_data['output_tensor'][-1]
            mass_conc = final_window[0]

        plot_data[(rec['beta_c'], rec['diffc'])] = {
            'mass_conc': mass_conc,
            'scenario': rec['scenario'],
            'scenario_index': rec['scenario_index'],
        }
        vmax = max(vmax, float(np.nanmax(mass_conc)))

    run_label.value = f'<b>Selected run:</b> {selected_run}'

    with plot_output:
        clear_output(wait=True)

        nrows, ncols = len(beta_vals), len(diff_vals)
        fig, axes = plt.subplots(
            nrows, ncols,
            figsize=(4.2 * ncols, 1.8 * nrows),
            squeeze=False
        )

        im = None
        for i, beta_c in enumerate(beta_vals):
            for j, diffc in enumerate(diff_vals):
                ax = axes[i][j]
                cell = plot_data.get((beta_c, diffc))

                if cell is None:
                    ax.axis('off')
                    continue

                im = ax.imshow(cell['mass_conc'], cmap='Reds', vmin=0, vmax=vmax)
                ax.set_xticks([])
                ax.set_yticks([])

                if i == 0:
                    ax.set_title(f'diffc = {diffc:.4f}')
                if j == 0:
                    ax.set_ylabel(f'beta_c = {beta_c:.2f}\n{cell["scenario"]}', rotation=90)

        if im is not None:
            fig.subplots_adjust(right=0.88)
            cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
            fig.colorbar(im, cax=cax, label='Mass concentration')

        inflow_values = []
        for info in scenario_map.values():
            run_meta = info.get('run_meta_by_name', {}).get(selected_run, {})
            inflow = run_meta.get('inflow', run_meta.get('inflow_value'))
            if inflow is not None:
                inflow_values.append(inflow)

        inflow_text = f" | inflow: {inflow_values[0]}" if inflow_values else ""
        fig.suptitle(f'Mass concentration grid for {selected_run}{inflow_text}', fontsize=24, y=0.98)
        plt.show()

run_dropdown.observe(render_plot, names='value')

controls = widgets.VBox([run_dropdown, run_label])
display(controls, plot_output)
render_plot()

Output()

In [ ]:
# Scenario profile viewer (with timestep slider + deduplicated renders).
scenario_options = sorted(scenario_map.keys())
if not scenario_options:
    raise ValueError('No scenarios available in scenario_map.')

if not common_runs:
    raise ValueError('No run options available in common_runs.')

scenario_profile_dropdown = widgets.Dropdown(
    options=scenario_options,
    value=scenario_options[0],
    description='Scenario:',
    layout=widgets.Layout(width='55%')
)

run_profile_dropdown = widgets.Dropdown(
    options=common_runs,
    value=run_dropdown.value if run_dropdown.value in common_runs else common_runs[0],
    description='Run:',
    layout=widgets.Layout(width='55%')
)

timestep_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description='Timestep:',
    continuous_update=False,
    layout=widgets.Layout(width='55%')
)

scenario_profile_output = widgets.Output()
render_state = {'last_key': None, 'busy': False}

def _current_windows_path():
    return os.path.join(
        scenarios_root,
        scenario_profile_dropdown.value,
        run_profile_dropdown.value,
        'windows.npz'
    )

def _refresh_timestep_bounds(*_):
    windows_path = _current_windows_path()
    max_idx = 0

    if os.path.exists(windows_path):
        with np.load(windows_path) as data:
            nsteps = int(data['output_tensor'].shape[0])
        max_idx = max(0, nsteps - 1)

    timestep_slider.max = max_idx
    if timestep_slider.value > max_idx:
        timestep_slider.value = max_idx

def render_scenario_profile(*_):
    scenario_key = scenario_profile_dropdown.value
    selected_run = run_profile_dropdown.value
    t_idx = timestep_slider.value
    key = (scenario_key, selected_run, t_idx)

    if render_state['busy'] or key == render_state['last_key']:
        return

    render_state['busy'] = True
    render_state['last_key'] = key

    windows_path = _current_windows_path()
    try:
        with scenario_profile_output:
            clear_output(wait=True)

            if not os.path.exists(windows_path):
                print(f'No data found: {windows_path}')
                return

            with np.load(windows_path) as data:
                output_tensor = data['output_tensor']
                nsteps = int(output_tensor.shape[0])
                t_idx = min(timestep_slider.value, max(0, nsteps - 1))
                if timestep_slider.value != t_idx:
                    timestep_slider.value = t_idx
                conc = output_tensor[t_idx][0]

                input_tensor = data['input_tensor'] if 'input_tensor' in data else None
                in_conc = None
                if input_tensor is not None:
                    if input_tensor.ndim >= 4:
                        in_conc = input_tensor[t_idx][0]
                    elif input_tensor.ndim == 3:
                        in_conc = input_tensor[t_idx]
                    else:
                        in_conc = np.squeeze(input_tensor)

                diff = in_conc - conc if in_conc is not None else np.full_like(conc, np.nan)

            ny, nx = conc.shape
            y_lines = [ny // 4, ny // 2, (3 * ny) // 4]
            colors = ['blue', 'green', 'orange']
            x = np.arange(nx)

            fig, axes = plt.subplots(1, 5, figsize=(28, 5), constrained_layout=True)

            ax0 = axes[0]
            im = ax0.imshow(conc, cmap='Reds', origin='upper', aspect='auto')
            for y_idx, c in zip(y_lines, colors):
                ax0.axhline(y_idx, color=c, linestyle='--', linewidth=2)
            ax0.set_title(f'{scenario_key} | {selected_run} | t={t_idx}')
            ax0.set_xlabel('x')
            ax0.set_ylabel('y')
            fig.colorbar(im, ax=ax0, label='Output concentration')

            for ax, y_idx, c in zip(axes[1:4], y_lines, colors):
                ax.plot(x, conc[y_idx, :], color=c, linewidth=2)
                ax.set_title(f'Output profile (y = {y_idx})')
                ax.set_xlabel('x')
                ax.set_ylabel('Concentration')
                ax.grid(True, alpha=0.3)
                # ax.set_ylim(0, 35.0)

            ax4 = axes[4]
            if np.isnan(diff).all():
                ax4.text(0.5, 0.5, 'input_tensor not found', ha='center', va='center', transform=ax4.transAxes)
                ax4.set_title('Input - Output')
                ax4.set_axis_off()
            else:
                vmax = np.nanmax(np.abs(diff))
                im_diff = ax4.imshow(
                    diff,
                    cmap='coolwarm',
                    origin='upper',
                    aspect='auto',
                    vmin=-vmax,
                    vmax=vmax
                )
                ax4.set_title('Input - Output')
                ax4.set_xlabel('x')
                ax4.set_ylabel('y')
                fig.colorbar(im_diff, ax=ax4, label='Δ concentration')

            plt.show()
            plt.close(fig)
    finally:
        render_state['busy'] = False

def _on_selection_change(*_):
    _refresh_timestep_bounds()
    render_state['last_key'] = None
    render_scenario_profile()

scenario_profile_dropdown.observe(_on_selection_change, names='value')
run_profile_dropdown.observe(_on_selection_change, names='value')
timestep_slider.observe(render_scenario_profile, names='value')

clear_output(wait=True)
display(
    widgets.VBox([scenario_profile_dropdown, run_profile_dropdown, timestep_slider]),
    scenario_profile_output
)

_refresh_timestep_bounds()
render_state['last_key'] = None
render_scenario_profile()

Output()

In [5]:
# # Source is the new unified scenarios directory
# src_scenarios_dir = scenarios_root  # already defined as base_dir/scenarios

# # Destination should NOT be the same as source/base_dir
# clean_dirty = os.path.expanduser('~/Projects/groundwater/data/henry_data/grid_scenarios_20x40_clean/scenarios')

# import shutil

# def copy_scenarios_exclude_hds_ucn(src_dir, dst_dir):
#     os.makedirs(dst_dir, exist_ok=True)

#     for root, dirs, files in os.walk(src_dir):
#         # Skip hidden directories
#         dirs[:] = [d for d in dirs if not d.startswith('.')]

#         rel_path = os.path.relpath(root, src_dir)
#         dst_root = os.path.normpath(os.path.join(dst_dir, rel_path))
#         os.makedirs(dst_root, exist_ok=True)

#         for file in files:
#             # Skip hidden files and unwanted binaries
#             if file.startswith('.') or file.endswith(('.hds', '.ucn')):
#                 continue

#             src_file = os.path.normpath(os.path.join(root, file))
#             dst_file = os.path.normpath(os.path.join(dst_root, file))

#             # Guard against accidental same-file copies
#             if os.path.exists(dst_file) and os.path.samefile(src_file, dst_file):
#                 continue

#             shutil.copy2(src_file, dst_file)

# copy_scenarios_exclude_hds_ucn(src_scenarios_dir, clean_dirty)
